# Arena 3D Reconstruction with Gaussian Splatting

Reconstruct a 10-15m arena from 91 photos using COLMAP + 3D Gaussian Splatting,
then compress the model for lightweight local viewing.

## Pipeline Overview (run cells in order)

| Step | Cell | What it does | Time | GPU? |
|------|------|-------------|------|------|
| 1 | **Cell 1** | Mount Google Drive | ~30s | No |
| 2 | **Cell 2** | Install COLMAP, PyTorch, CUDA extensions | ~3 min | No* |
| 3 | **Cell 3** | Load images from a folder of your choice | ~2 min | No |
| 4 | **Cell 4A-4D** | COLMAP SfM: features, matching, reconstruction, merge | ~25 min | No |
| 5 | **Cell 5** | Verify pre-computed COLMAP (local path) | ~1 min | No |
| 6 | **Cell 6** | Convert COLMAP to 3DGS format | ~1 min | No |
| 7 | **Cell 7A-7B** | Train 3D Gaussian Splatting (enhanced, gsplat-based) | 7-30 min | **Yes (T4+)** |
| 8 | **Cell 8A** | Export final point cloud PLY | ~1 min | No |
| 9 | **Cell 8B** | Validate PLY for Unity + viewing options | ~1 min | No |
| 10 | **Cell 8C** | Compress model for local decompression viewer | ~1 min | No |

## Outputs

| File | Location | Size |
|------|----------|------|
| Final model (PLY) | `arena_3dgs_pointcloud.ply` | ~100-500 MB |
| Compressed model (.splat) | `arena_3dgs_compressed.splat` | ~10-50 MB |
| Training output (PLY) | `output/arena_3dgs/arena_3dgs.ply` | ~100-500 MB |


In [1]:
#@title === 1. Mount Google Drive ===
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PATH = "/content/drive/MyDrive/"
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Drive ready: {DRIVE_PATH}")


Mounted at /content/drive
Drive ready: /content/drive/MyDrive/


In [2]:
#@title === 2. Install Dependencies (~3 min, idempotent) ===
import os, sys, urllib.request
SCRIPTS_DIR = "/content/scripts"
os.makedirs(SCRIPTS_DIR, exist_ok=True)
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

sys.modules.pop("scripts.colab_pipeline", None)
url = "https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/colab_pipeline.py"
urllib.request.urlretrieve(url, os.path.join(SCRIPTS_DIR, "colab_pipeline.py"))
print("Downloaded latest colab_pipeline.py from GitHub")

from scripts.colab_pipeline import install_dependencies
install_dependencies(drive_path=DRIVE_PATH, scripts_dir=SCRIPTS_DIR)


Downloaded latest colab_pipeline.py from GitHub
[1/4] Installing COLMAP + display deps...
[2/4] Installing PyTorch...
  PyTorch 2.11.0+cu128, CUDA: True, VRAM: 15.6GB
[3/4] Installing Python packages...
[4/4] Verifying GPU...

System deps installed!
  Done.
  Done.

All dependencies ready!


---
## Optional: Extract images from a ZIP file

If your images are packed in a ZIP on Google Drive, run this cell first.
It extracts them to a folder you can then point to in Step 3.
Skip this if your images are already extracted.
---


In [3]:
#@title === Optional: Extract ZIP to a folder ===
import zipfile, os

ZIP_PATH = "/content/drive/MyDrive/tandt_db.zip" #@param {type:"string"}
EXTRACT_TO = "/content/splats3d" #@param {type:"string"}

os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_TO)

extracted = [f for f in os.listdir(EXTRACT_TO) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"Extracted {len(extracted)} images to {EXTRACT_TO}")
print(f"Use this path in Step 3 below.")


Extracted 0 images to /content/splats3d
Use this path in Step 3 below.


---
## Step 3: Images

Point to a folder of images on your Google Drive.  They will be copied to
the training directory.  Supported formats: .jpg, .jpeg, .png
---


In [4]:
#@title === 3: Load Images ===
IMAGE_PATH = "/content/splats3d/tandt/train/images" #@param {type:"string"}

from scripts.colab_pipeline import load_images
load_images(IMAGE_PATH)


Copied 301 images from /content/splats3d/tandt/train/images


---
## COLMAP Step 4: Structure from Motion

Four modular cells: (A) features, (B) matching, (C) reconstruction, (D) merge.

Paths are defined once in **4A** and reused by 4B-4D.

**Alternatively**, skip to Step 5 to use pre-computed camera poses.
---


In [13]:
#@title === 4A: Feature Extraction (~5 min, idempotent) ===
# --- Shared paths (used by 4A-4D, 5, 6, 7) ---
INPUT_DIR = "/content/gaussian-splatting/input" #@param {type:"string"}
DB_PATH = "/content/gaussian-splatting/sparse/database.db" #@param {type:"string"}
COLMAP_DIR = "/content/gaussian-splatting/sparse" #@param {type:"string"}
SPARSE_DIR = "/content/gaussian-splatting/input/sparse/0" #@param {type:"string"}
MERGED_DIR = "/content/gaussian-splatting/sparse_merged" #@param {type:"string"}

# --- 4A params ---
CAMERA_MODEL = "SIMPLE_RADIAL" #@param {type:"string"}
SINGLE_CAMERA = True #@param {type:"boolean"}
MAX_NUM_FEATURES = 8192 #@param {type:"integer"}
FIRST_OCTAVE = -1 #@param {type:"integer"}
PEAK_THRESHOLD = 0.01 #@param {type:"number"}

from scripts.colab_pipeline import run_colmap_features
run_colmap_features(
    input_dir=INPUT_DIR,
    colmap_dir=COLMAP_DIR,
    db_path=DB_PATH,
    camera_model=CAMERA_MODEL,
    single_camera=SINGLE_CAMERA,
    max_num_features=MAX_NUM_FEATURES,
    first_octave=FIRST_OCTAVE,
    peak_threshold=PEAK_THRESHOLD,
)


Features already extracted. Skipping.


In [14]:
#@title === 4B: Feature Matching (~5 min, idempotent) ===
MATCHING_MODE = "exhaustive only" #@param ["sequential + exhaustive", "sequential only", "exhaustive only"]
SEQUENTIAL_OVERLAP = 20 #@param {type:"integer"}

from scripts.colab_pipeline import run_colmap_matching
run_colmap_matching(
    db_path=DB_PATH,
    matching_mode=MATCHING_MODE,
    sequential_overlap=SEQUENTIAL_OVERLAP,
)


3675 match pairs already exist. Skipping.


In [15]:
#@title === 4C: COLMAP Reconstruction (~10 min, idempotent) ===
MULTIPLE_MODELS = True #@param {type:"boolean"}
MAX_NUM_MODELS = 50 #@param {type:"integer"}
INIT_MIN_TRI_ANGLE = 4 #@param {type:"number"}
INIT_MIN_NUM_INLIERS = 15 #@param {type:"integer"}
ABS_POSE_MIN_NUM_INLIERS = 8 #@param {type:"integer"}
BA_LOCAL_MAX_NUM_ITERATIONS = 25 #@param {type:"integer"}
BA_GLOBAL_MAX_NUM_ITERATIONS = 50 #@param {type:"integer"}

from scripts.colab_pipeline import run_colmap_reconstruction
run_colmap_reconstruction(
    input_dir=INPUT_DIR,
    colmap_dir=COLMAP_DIR,
    db_path=DB_PATH,
    sparse_dir=SPARSE_DIR,
    multiple_models=MULTIPLE_MODELS,
    max_num_models=MAX_NUM_MODELS,
    init_min_tri_angle=INIT_MIN_TRI_ANGLE,
    init_min_num_inliers=INIT_MIN_NUM_INLIERS,
    abs_pose_min_num_inliers=ABS_POSE_MIN_NUM_INLIERS,
    ba_local_max_num_iterations=BA_LOCAL_MAX_NUM_ITERATIONS,
    ba_global_max_num_iterations=BA_GLOBAL_MAX_NUM_ITERATIONS,
)


Model already exists (112 images). Skipping.

Best model: sub=0, images=112


In [16]:
#@title === 4D: Model Merging (~2 min, idempotent) ===
MIN_IMAGES_FOR_MERGE = 5 #@param {type:"integer"}

from scripts.colab_pipeline import run_colmap_merge
run_colmap_merge(
    colmap_dir=COLMAP_DIR,
    merged_dir=MERGED_DIR,
    sparse_dir=SPARSE_DIR,
    min_images_for_merge=MIN_IMAGES_FOR_MERGE,
)


  Found model 0: 112 images
Not enough models to merge.

Optimized COLMAP result: 112 images, 16340 3D points


---
## Step 5: Verify Pre-computed COLMAP Data (~1 min)

Skip COLMAP (Step 4) and use pre-computed camera poses from a local path
containing cameras.txt, images.txt, points3D.txt.
---


In [17]:
#@title === 5: Verify Pre-computed COLMAP Data (~1 min, idempotent) ===
SPARSE_DIR = "/content/gaussian-splatting/sparse/0" #@param {type:"string"}

from scripts.colab_pipeline import verify_sparse_model
verify_sparse_model(sparse_dir=SPARSE_DIR)


FileNotFoundError: Missing COLMAP data in /content/gaussian-splatting/sparse/0: ['cameras.txt', 'images.txt', 'points3D.txt']

---
## Step 6: Convert to 3DGS Format (~1 min)

Prepares the COLMAP output (sparse model + images) for the 3DGS trainer.
---


In [ ]:
#@title === 6: Convert Data to 3DGS Format (~1 min, idempotent) ===
# images_dir is derived from input_dir

IMAGES_DIR = "/content/gaussian-splatting/FanTest_3dgs_input/images"
INPUT_DIR=IMAGES_DIR

from scripts.colab_pipeline import convert_to_3dgs_format
convert_to_3dgs_format(
    sparse_dir=SPARSE_DIR,
    input_dir=IMAGES_DIR,
    images_dir=IMAGES_DIR,
)



Ready for training: 29 images


---
## Step 7: Train 3D Gaussian Splatting

Training uses the enhanced gsplat-based script for faster, more memory-efficient
training without CUDA extension compilation.

**Run order:**
1. Cell 7A: Quick test (3K iters, ~7 min) - verify everything works
2. Cell 7B: Full training (30K iters, ~30 min) - main training run
---


In [ ]:
#@title === 7A: Quick Test (3000 iters, ~7 min) ===
TEST_ITERS = 3000 #@param {type:"integer"}
TEST_MAX_GAUSSIANS = 0 #@param {type:"integer"}
TEST_LOG_INTERVAL = 500 #@param {type:"integer"}
TEST_MAX_RES = 800 #@param {type:"integer"}
TEST_RANDOM_BG = False #@param {type:"boolean"}
TEST_OPACITY_RESET = 3000 #@param {type:"integer"}
TEST_DENSIFY_UNTIL = 1500 #@param {type:"integer"}
TEST_SH_DEGREE = 1 #@param {type:"integer"}
TEST_SH_DEGREE_INTERVAL = 500 #@param {type:"integer"}
TEST_FORCE_SPLIT_SCALE = 0.02 #@param {type:"number"}

from scripts.colab_pipeline import train_3dgs
OUTPUT_BASE = "/content/gaussian-splatting/output" #@param {type:"string"}
train_3dgs(
    input_dir=INPUT_DIR,
    sparse_dir=SPARSE_DIR,
    output_base=OUTPUT_BASE,
    iterations=TEST_ITERS,
    max_gaussians=TEST_MAX_GAUSSIANS,
    log_interval=TEST_LOG_INTERVAL,
    max_res=TEST_MAX_RES,
    output_name="quick_test",
    random_background=TEST_RANDOM_BG,
    opacity_reset_interval=TEST_OPACITY_RESET,
    densify_until_iter=TEST_DENSIFY_UNTIL,
    sh_degree=TEST_SH_DEGREE,
    sh_degree_interval=TEST_SH_DEGREE_INTERVAL,
    force_split_scale=TEST_FORCE_SPLIT_SCALE,
)


Using device: cuda
Images: /content/gaussian-splatting/FanTest_3dgs_input/images
Sparse: /content/gaussian-splatting/FanTest_3dgs_input/sparse/0
Loading COLMAP data...
  1 cameras, 29 images, 631 points
  Loaded 29 training views
  Downscaling images: 1080x1920 → 900x1600 (scale=0.833)
  Initialized 631 Gaussians

ERROR: Training failed: cannot access local variable 'active_sh_degree' where it is not associated with a value


Traceback (most recent call last):
  File "/content/scripts/colab_pipeline.py", line 428, in train_3dgs
    train_fn(args)
  File "/content/scripts/train_3dgs_enhanced.py", line 452, in train
    'active_sh_degree': active_sh_degree,
                        ^^^^^^^^^^^^^^^^
UnboundLocalError: cannot access local variable 'active_sh_degree' where it is not associated with a value


In [ ]:
#@title === 7B: Full Training 30K (~30 min, single run) ===
FULL_ITERS = 30000 #@param {type:"integer"}
FULL_MAX_GAUSSIANS = 350000 #@param {type:"integer"}
FULL_DENSIFY_UNTIL = 15000 #@param {type:"integer"}
FULL_OPACITY_RESET = 3000 #@param {type:"integer"}
FULL_RANDOM_BG = False #@param {type:"boolean"}
FULL_SH_DEGREE = 3 #@param {type:"integer"}
FULL_SH_DEGREE_INTERVAL = 1000 #@param {type:"integer"}
FULL_FORCE_SPLIT_SCALE = 0.02 #@param {type:"number"}
FULL_LOG_INTERVAL = 1000 #@param {type:"integer"}

from scripts.colab_pipeline import train_3dgs
train_3dgs(
    input_dir=INPUT_DIR,
    sparse_dir=SPARSE_DIR,
    output_base=OUTPUT_BASE,
    iterations=FULL_ITERS,
    max_gaussians=FULL_MAX_GAUSSIANS,
    log_interval=FULL_LOG_INTERVAL,
    max_res=800,
    output_name="arena_3dgs",
    random_background=FULL_RANDOM_BG,
    opacity_reset_interval=FULL_OPACITY_RESET,
    densify_until_iter=FULL_DENSIFY_UNTIL,
    sh_degree=FULL_SH_DEGREE,
    sh_degree_interval=FULL_SH_DEGREE_INTERVAL,
    force_split_scale=FULL_FORCE_SPLIT_SCALE,
)


---
## Step 8: Export & Validate for Unity (~1 min)
---


In [ ]:
#@title === 8A: Export Point Cloud (~1 min) ===
OUTPUT_DIRS = "/content/gaussian-splatting/output/arena_3dgs/arena_3dgs.ply" #@param {type:"string"}
EXPORT_PLY_NAME = "arena_3dgs_pointcloud.ply" #@param {type:"string"}

from scripts.colab_pipeline import export_pointcloud
export_pointcloud(
    output_dirs=[OUTPUT_DIRS],
    dst=f"/content/{EXPORT_PLY_NAME}",
)


In [ ]:
#@title === 8B: Validate PLY for Unity (~1 min) ===
PLY_TO_VALIDATE = "/content/arena_3dgs_pointcloud.ply" #@param {type:"string"}

from scripts.colab_pipeline import validate_pointcloud
import os
validate_pointcloud(PLY_TO_VALIDATE)

print("\n" + "=" * 50)
print("  VIEWING OPTIONS")
print("=" * 50)
print("\n1. SuperSplat (no install, web):")
print("     https://supersplat.com/")
print("\n2. Unity walkthrough (best):")
print("     Clone: https://github.com/aras-p/UnityGaussianSplatting")
print("     Drop PLY into Assets/GaussianAssets/")
print("     WASD + mouse-look controls")
print("\n3. Local decompression viewer (lightweight, no GPU):")
print("     python3 scripts/decompress_splat.py compressed.splat")
print("     Drag to orbit, scroll to zoom, R=reset, Q=quit")


In [ ]:
#@title === 8C: Compress Model for Local Viewer (~1 min) ===
PLY_PATH = "/content/arena_3dgs_pointcloud.ply" #@param {type:"string"}
COMPRESS_QUALITY = "medium" #@param {type:"string"}

import urllib.request
import subprocess, sys, os, glob

if not os.path.exists(PLY_PATH):
    print(f"PLY not found: {PLY_PATH}. Run 8A first.")
else:
    SCRIPT = "/content/compress_splat.py"
    if not os.path.exists(SCRIPT):
        url = "https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/compress_splat.py"
        urllib.request.urlretrieve(url, SCRIPT)
        print("Downloaded compress_splat.py")

    quality = COMPRESS_QUALITY

    print("Exporting standard .splat (SuperSplat-compatible)...")
    subprocess.run(
        [sys.executable, SCRIPT, PLY_PATH, "--standard", "--output-dir", "/content"],
        capture_output=False
    )

    print(f"Running compression (quality={quality})...")
    result = subprocess.run(
        [sys.executable, SCRIPT, PLY_PATH, "--quality", quality, "--output-dir", "/content"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)

    splats = glob.glob("/content/*.splat")
    if splats:
        for splat in splats:
            size_mb = os.path.getsize(splat) / (1024 * 1024)
            print(f"  {splat} ({size_mb:.1f} MB)")
        print("\nDownload the .splat files to your computer:")
        print("  *_standard.splat -> drag into https://supersplat.com/editor (no install)")
        print("  *_compressed.splat -> python3 scripts/decompress_splat.py path/to/file")
        from google.colab import files
        for splat in splats:
            files.download(splat)


In [ ]:
#@title === 8D: Visualize Initial Point Cloud Coverage (~1 min) ===
PLY_TO_CHECK = "/content/gaussian-splatting/output/quick_test/arena_3dgs_init.ply" #@param {type:"string"}
SPARSE_DIR_CHECK = "/content/gaussian-splatting/FanTest_3dgs_input/sparse/0" #@param {type:"string"}
IMAGES_DIR_CHECK = "/content/gaussian-splatting/FanTest_3dgs_input/images" #@param {type:"string"}

from scripts.visualize_coverage import check_coverage
check_coverage(
    sparse_dir=SPARSE_DIR_CHECK,
    ply_path=PLY_TO_CHECK,
    images_dir=IMAGES_DIR_CHECK,
    output_dir="/content/coverage_views",
)
print("\nOpen /content/coverage_views/ to see each view with projected points.")
print("Views with few green dots lack initial Gaussian coverage.")


---
## Appendix: Troubleshooting

| Problem | Solution |
|---------|----------|
| **train_3dgs_enhanced.py not found** | Re-run Cell 2 to download from GitHub |
| **CUDA out of memory** | Reduce `max_gaussians` (e.g. 200000) or `max_res` (e.g. 1200) in training cell |
| **COLMAP produces 0 images** | Use pre-computed data (Step 5) |
| **Drive missing** | Ensure `MyDrive/arena_3dgs/` exists |

### Checkpoint locations:
- `output/arena_3dgs/arena_3dgs.ply` - Trained model PLY file
- `/content/arena_3dgs_pointcloud.ply` - Final PLY export

---
